In [ ]:
# SET UP AND LIBRARIES

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.stats import pearsonr


plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Libraries loaded successfully. Ready for Data Analysis.")

In [ ]:

#LOWPASS FILTER AND BEGGINING AND END CLIPPER


def butter_lowpass_filter(data, cutoff, fs, order=4):
    """
    Applies a zero-phase low-pass Butterworth filter to remove high-frequency IMU noise.
    :param data: The raw angle array.
    :param cutoff: Cutoff frequency in Hz (e.g. 5Hz - standard for human movement).
    :param fs: Sampling frequency in Hz (60Hz for Unity).
    :param order: order of the filter.
    """
    nyq = 0.5 * fs # Nyquist Frequency
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

def trim_dead_time(df, angle_column, velocity_threshold=1.0):
    """
    Removes flat lines at the beginning and end of the recording.
    Uses angular velocity to detect movement onset and offset.
    """
    # Calculate angular velocity
    velocity = np.abs(np.gradient(df[angle_column]))

    # Find indices where velocity exceeds the threshold
    movement_indices = np.where(velocity > velocity_threshold)[0]

    if len(movement_indices) == 0:
        print("No movement detected.")
        return df # Return original if no movement found

    start_idx = movement_indices[0]
    end_idx = movement_indices[-1]

    buffer = 10
    start_idx = max(0, start_idx - buffer)
    end_idx = min(len(df) - 1, end_idx + buffer)

    trimmed_df = df.iloc[start_idx:end_idx].copy()

    # Reset the time so the trimmed clip starts at Time = 0
    trimmed_df['Time(s)'] = trimmed_df['Time(s)'] - trimmed_df['Time(s)'].iloc[0]

    return trimmed_df


def generate_ideal_curve(time_array, target_min, target_max, frequency_hz):
    """
    Generates the theoretical 'Ideal Curve' based on the metronome pace.
    Assumes a continuous cyclical movement (sine wave).
    """
    amplitude = (target_max - target_min) / 2
    offset = target_min + amplitude

    # Generate a sine wave matching the metronome frequency
    ideal_wave = offset - amplitude * np.cos(2 * np.pi * frequency_hz * time_array)
    return ideal_wave

In [ ]:
#FILE ANALYSIS AND PLOTTING

# Configuration
file_path = 'S01_Elbow_Continuous_Flexion_Normal.csv'

# Axis to analyze (Flexion is usually Z or X, check your data)
target_axis = 'Elbow_Z'

# Metronome config (e.g., if metronome is 60 BPM, 1 full rep (up and down) takes 2 beats = 0.5 Hz)
movement_freq_hz = 0.5
target_angle_min = 0.0   # Arm resting
target_angle_max = 90.0  # Full flexion target

# LOAD AND CLEAN DATA

df_raw = pd.read_csv(file_path)
df_trimmed = trim_dead_time(df_raw, target_axis)

# Estimate sampling rate (FPS) from the timestamps
time_diffs = np.diff(df_trimmed['Time(s)'])
fs_estimated = 1.0 / np.mean(time_diffs)

# Apply Low-Pass Filter (Cutoff at 3Hz is very aggressive/smooth, 5Hz is standard)
df_trimmed['Filtered_Angle'] = butter_lowpass_filter(df_trimmed[target_axis], cutoff=5.0, fs=fs_estimated)

# Generate Ideal Curve
time_array = df_trimmed['Time(s)'].values
ideal_curve = generate_ideal_curve(time_array, target_angle_min, target_angle_max, movement_freq_hz)

# CALCULATE METRICS
unity_curve = df_trimmed['Filtered_Angle'].values

# RMSE (Root Mean Square Error)
rmse = np.sqrt(np.mean((unity_curve - ideal_curve)**2))

# Peak Error (Maximum absolute difference)
peak_error = np.max(np.abs(unity_curve - ideal_curve))

# Pearson Correlation (r)
pearson_r, _ = pearsonr(unity_curve, ideal_curve)

print(f"--- METRICS FOR {file_path} ---")
print(f"RMSE:          {rmse:.2f}º")
print(f"Peak Error:    {peak_error:.2f}º")
print(f"Pearson (r):   {pearson_r:.4f}")

# VISUALIZATION
plt.figure()

# Plot the curves
plt.plot(df_trimmed['Time(s)'], df_trimmed[target_axis], color='lightgray', label='Raw Unity Data', alpha=0.6)
plt.plot(df_trimmed['Time(s)'], unity_curve, color='#1f77b4', label='Filtered Unity Data (IMU)', linewidth=2.5)
plt.plot(df_trimmed['Time(s)'], ideal_curve, color='#d62728', linestyle='--', label='Ideal Target (Metronome)', linewidth=2)


plt.title(f'Kinematic Validation: {file_path.replace(".csv", "")}', pad=15, fontweight='bold')
plt.xlabel('Time (seconds)', fontweight='bold')
plt.ylabel('Angle (degrees)', fontweight='bold')
plt.legend(loc='upper right', frameon=True, shadow=True)

# Save the plot
output_image_name = file_path.replace('.csv', '_plot.png')
plt.savefig(output_image_name, dpi=300, bbox_inches='tight')
print(f"Plot saved successfully as: {output_image_name}")

plt.show()